# V11 — Patient-Specific Leave-One-Seizure-Out (LOSO)

**Why this notebook exists.** Published seizure-prediction studies that report
high sensitivity (Truong et al. 2018: Sens = 0.81; Khan et al. 2018: 0.875;
Bandarabadi et al. 2015: 0.79) all use **patient-specific** evaluation, not
cross-patient LOPO. Under their protocol, every patient gets a personalised
model trained on that patient's own historical seizures, and evaluation
rotates the held-out seizure. Without that distinction, the cross-patient
LOPO numbers in V4b / V5c / V6b cannot be compared to published headlines.

This notebook implements the exact Truong-style protocol:

  For each patient p with N_p annotated seizures:
      For each held-out seizure k ∈ {1, ..., N_p}:
          - Train on preictal windows preceding all OTHER seizures of p
            (plus balanced interictal sample)
          - Evaluate on the held-out seizure's preictal window and the
            adjacent interictal segment
          - Apply literature-standard post-processing
            (smoothing → operational threshold → K-of-M vote + refractory)

Aggregation:
  - Per-fold (per-seizure) sensitivity = was the held-out seizure correctly
    flagged by ≥ 1 alarm during its preictal SOP?
  - Per-patient FPR/h = mean false predictions per interictal hour
  - Per-patient mean ± std across the N_p folds, then across-patient mean

Features: PDC VAR(20) all-band from `cache_pdc_var20/` (no recomputation).
Classifiers: LR, SVM (matching V6b for direct comparison).


In [1]:
# --- portable repo bootstrap (added for public release; locates the repo root) ---
import sys as _sys, pathlib as _pl
REPO = _pl.Path.cwd()
while not (REPO / 'src' / 'config.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
_sys.path.insert(0, str(REPO / 'src'))
from pathlib import Path
CODE_DIR = str(REPO); CODE = REPO; CODEV2 = REPO; PROJECT_DIR = REPO
# --------------------------------------------------------------------------------

# Cell 0 — Imports & config
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# [path set by bootstrap] CODE_DIR = r"<repo>/Code"
sys.path.insert(0, CODE_DIR)
from config import (
    DATA_ROOT, STEP_SEC, EXCLUDED_PATIENTS, RESULTS_DIR, RANDOM_SEED,
    ALARM_K, ALARM_M, ALARM_REFRACTORY,
)
from metrics import evaluate_with_alarms

import importlib, alarm_postproc
importlib.reload(alarm_postproc)
from alarm_postproc import smooth_probs, operational_threshold

from sklearn.linear_model    import LogisticRegression
from sklearn.svm             import SVC
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import roc_curve, roc_auc_score, average_precision_score

np.random.seed(RANDOM_SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)

SMOOTH_WINDOW = 10
TARGET_FPR_H  = 1.0
print(f'V11 — LOSO patient-specific  K={ALARM_K}/M={ALARM_M}, smooth={SMOOTH_WINDOW}w, FPR/h≤{TARGET_FPR_H}')


V11 — LOSO patient-specific  K=5/M=12, smooth=10w, FPR/h≤1.0


In [2]:
# Cell 1 — Load cached PDC features (chronological order preserved)
PDC_CACHE = Path(CODE_DIR) / 'cache_pdc_var20'
assert PDC_CACHE.exists()

patients_all = sorted([
    p for p in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, p))
    and p.startswith('chb') and p not in EXCLUDED_PATIENTS
])

pdc_data = {}
for pid in patients_all:
    pdir = PDC_CACHE / pid
    feat_p = pdir / 'features.npy'
    lab_p  = pdir / 'labels.npy'
    if not (feat_p.exists() and lab_p.exists()): continue
    X = np.load(feat_p).astype(np.float32)
    y = np.load(lab_p).astype(np.int8)
    if (y==1).sum() < 3: continue
    pdc_data[pid] = (X, y)

patient_ids = sorted(pdc_data.keys())
print(f'Loaded {len(patient_ids)} patients.')


Loaded 21 patients.


In [3]:
# Cell 2 — Identify seizure events from preictal block boundaries
def identify_seizures(y):
    """
    Return [(start, end), ...] for each contiguous preictal block (y==1).
    Each block corresponds to one annotated seizure event.
    """
    in_block = False
    starts, ends = [], []
    for i, v in enumerate(y):
        if v == 1 and not in_block:
            starts.append(i); in_block = True
        elif v != 1 and in_block:
            ends.append(i); in_block = False
    if in_block:
        ends.append(len(y))
    return list(zip(starts, ends))

# Inventory: how many seizures per patient?
print(f'{"Patient":<8} {"#seizures":>10} {"preictal":>10} {"interictal":>11}')
print('-' * 45)
seizure_inventory = {}
for pid in patient_ids:
    X, y = pdc_data[pid]
    blocks = identify_seizures(y)
    seizure_inventory[pid] = blocks
    print(f'{pid:<8} {len(blocks):>10}  {int((y==1).sum()):>10}  {int((y==0).sum()):>11}')

# Patients with at least 2 seizures are eligible for LOSO
eligible = [p for p, b in seizure_inventory.items() if len(b) >= 2]
print(f'\nEligible patients (≥2 seizures): {len(eligible)}/{len(patient_ids)}')


Patient   #seizures   preictal  interictal
---------------------------------------------
chb01             2         296         1037
chb02             2         296          339
chb03             3         444          899
chb04             2         296         2776
chb05             3         444          582
chb06             7        1037         6680
chb07             3         445         2488
chb08             5         741          535
chb09             4         592         2231
chb10             6         888         3062
chb13             3         444          947
chb14             6         774          687
chb15             3         320         1962
chb16             3         347          493
chb17             3         444          384
chb18             4         592          699
chb19             2         296          475
chb20             3         369          536
chb22             2         296          513
chb23             5         608         1382
chb24      

In [4]:
# Cell 3 — Patient-specific LOSO evaluation

def make_lr():
    return ('LR', Pipeline([('scl', StandardScaler()),
        ('clf', LogisticRegression(max_iter=400, solver='lbfgs',
            class_weight='balanced', C=1.0, random_state=RANDOM_SEED))]))

def make_svm():
    return ('SVM', Pipeline([('scl', StandardScaler()),
        ('clf', SVC(kernel='rbf', class_weight='balanced',
            C=1.0, probability=True, random_state=RANDOM_SEED))]))

MODEL_FACTORIES = [make_lr, make_svm]

def run_loso(factory, pdc_data, seizure_inventory, eligible):
    name, _ = factory()
    print(f'\n== {name} LOSO ({len(eligible)} patients, alarm-level FPR/h ≤ {TARGET_FPR_H}) ==')
    rows_fold = []           # per-(patient, held-out seizure) row
    rows_patient = []        # per-patient summary row
    t0 = time.time()

    for pi, pid in enumerate(eligible, 1):
        X, y = pdc_data[pid]
        blocks = seizure_inventory[pid]
        N = len(blocks)

        per_fold_metrics = []
        for k, (test_start, test_end) in enumerate(blocks):
            # Test window range: preictal block + the interictal segment
            # extending from the end of the previous seizure to the end of
            # this preictal block (Truong's "evaluation segment per held-out seizure").
            prev_end = blocks[k-1][1] if k > 0 else 0
            test_lo = prev_end
            test_hi = test_end          # ends exactly at end of preictal SOP

            test_idx = np.arange(test_lo, test_hi)
            train_idx = np.concatenate([
                np.arange(0, test_lo),
                np.arange(test_hi, len(y))
            ])
            if len(train_idx) < 50 or len(test_idx) < 30: continue
            X_tr, y_tr = X[train_idx], y[train_idx]
            X_te, y_te = X[test_idx],  y[test_idx]
            if len(np.unique(y_tr)) < 2 or len(np.unique(y_te)) < 2: continue

            _, pipe = factory()
            pipe.fit(X_tr, y_tr)
            probs_raw = pipe.predict_proba(X_te)[:, 1]

            # --- post-processing pipeline ---
            probs = smooth_probs(probs_raw, window=SMOOTH_WINDOW)
            threshold = operational_threshold(
                probs, y_te, STEP_SEC, target_fpr_h=TARGET_FPR_H,
                K=ALARM_K, M=ALARM_M, refractory_windows=ALARM_REFRACTORY,
            )

            a = evaluate_with_alarms(
                y_true=y_te, probs=probs, threshold=threshold,
                K=ALARM_K, M=ALARM_M, refractory_windows=ALARM_REFRACTORY,
                patient_id=pid,
            )
            row = {
                'patient': pid, 'seizure_idx': k, 'model': name,
                'auc': a['auc'], 'auc_pr': a['auc_pr'],
                'threshold': threshold,
                'sens_alarm': a['sensitivity'],
                'spec_alarm': a['specificity'],
                'fpr_per_hour': a['fpr_per_hour'],
                'n_pre_test': int((y_te==1).sum()),
                'n_int_test': int((y_te==0).sum()),
            }
            rows_fold.append(row)
            per_fold_metrics.append(row)

        if per_fold_metrics:
            df_pat = pd.DataFrame(per_fold_metrics)
            rows_patient.append({
                'patient':       pid, 'model': name,
                'n_seizures':    N,
                'auc_mean':      df_pat['auc'].mean(),
                'auc_pr_mean':   df_pat['auc_pr'].mean(),
                'sens_mean':     df_pat['sens_alarm'].mean(),
                'sens_event':    (df_pat['sens_alarm'] > 0).mean(),  # event-level: was each seizure flagged?
                'fpr_h_mean':    df_pat['fpr_per_hour'].mean(),
            })
            print(f'  [{pi:2d}/{len(eligible)}] {pid:<6} N={N}  '
                  f'AUC={df_pat["auc"].mean():.3f}  '
                  f'event-sens={(df_pat["sens_alarm"] > 0).mean():.2f}  '
                  f'FPR/h={df_pat["fpr_per_hour"].mean():.2f}')

    print(f'  {name} done in {(time.time()-t0)/60:.1f}min')
    return rows_fold, rows_patient

print('LOSO machinery ready.')


LOSO machinery ready.


In [5]:
# Cell 4 — Run LR and SVM under LOSO
results_fold, results_patient = {}, {}
for fac in MODEL_FACTORIES:
    name = fac()[0]
    rf, rp = run_loso(fac, pdc_data, seizure_inventory, eligible)
    results_fold[name]    = pd.DataFrame(rf)
    results_patient[name] = pd.DataFrame(rp)



== LR LOSO (21 patients, alarm-level FPR/h ≤ 1.0) ==
  [ 1/21] chb01  N=2  AUC=0.705  event-sens=1.00  FPR/h=0.40
  [ 2/21] chb02  N=2  AUC=0.409  event-sens=0.50  FPR/h=2.47
  [ 3/21] chb03  N=3  AUC=0.328  event-sens=0.33  FPR/h=1.35
  [ 4/21] chb04  N=2  AUC=0.228  event-sens=0.50  FPR/h=0.67
  [ 5/21] chb05  N=3  AUC=0.393  event-sens=0.67  FPR/h=0.31
  [ 6/21] chb06  N=7  AUC=0.577  event-sens=0.57  FPR/h=0.54
  [ 7/21] chb07  N=3  AUC=0.612  event-sens=0.33  FPR/h=0.55
  [ 8/21] chb08  N=5  AUC=0.477  event-sens=0.20  FPR/h=0.00
  [ 9/21] chb09  N=4  AUC=0.403  event-sens=0.25  FPR/h=0.30
  [10/21] chb10  N=6  AUC=0.621  event-sens=0.50  FPR/h=0.64
  [11/21] chb13  N=3  AUC=0.556  event-sens=0.33  FPR/h=0.53
  [12/21] chb14  N=6  AUC=0.355  event-sens=0.33  FPR/h=0.20
  [13/21] chb15  N=3  AUC=0.341  event-sens=0.00  FPR/h=1.35
  [14/21] chb16  N=3  AUC=0.260  event-sens=0.00  FPR/h=1.02
  [15/21] chb17  N=3  AUC=0.554  event-sens=1.00  FPR/h=1.56
  [16/21] chb18  N=4  AUC=0.604

In [6]:
# Cell 5 — Save and summarise
for name, df in results_fold.items():
    df.to_csv(Path(RESULTS_DIR) / f'lopo_v11_loso_fold_{name}.csv', index=False)
for name, df in results_patient.items():
    df.to_csv(Path(RESULTS_DIR) / f'lopo_v11_loso_patient_{name}.csv', index=False)

print(f'{"Model":<6} {"N pat.":>6} {"AUC":>6} {"AUC-PR":>7} {"Event-Sens":>10} {"FPR/h":>7}')
print('-' * 55)
for name, df in results_patient.items():
    print(f'{name:<6} {len(df):>6} {df.auc_mean.mean():>6.3f} {df.auc_pr_mean.mean():>7.3f} '
          f'{df.sens_event.mean():>10.3f} {df.fpr_h_mean.mean():>7.3f}')

print('\nEvent-sensitivity = fraction of held-out seizures correctly flagged')
print('                    by ≥ 1 alarm during the preictal SOP (Truong 2018 convention).')
print(f'\nLiterature reference: Truong 2018 Sens=0.81 @ FPR/h=0.16 (patient-specific)')
print(f'                      Khan   2018 Sens=0.875 @ FPR/h=0.142')
print(f'\nOutputs: lopo_v11_loso_{{fold,patient}}_{{LR,SVM}}.csv')


Model  N pat.    AUC  AUC-PR Event-Sens   FPR/h
-------------------------------------------------------
LR         21  0.486   0.451      0.419   0.751
SVM        21  0.449   0.427      0.406   0.464

Event-sensitivity = fraction of held-out seizures correctly flagged
                    by ≥ 1 alarm during the preictal SOP (Truong 2018 convention).

Literature reference: Truong 2018 Sens=0.81 @ FPR/h=0.16 (patient-specific)
                      Khan   2018 Sens=0.875 @ FPR/h=0.142

Outputs: lopo_v11_loso_{fold,patient}_{LR,SVM}.csv


## 6 · Three-tier protocol comparison

Headline table for the thesis Discussion: same PDC features, three evaluation
regimes, decreasing strictness, increasing realism for deployment.


In [7]:
# Cell 6 — Three-tier comparison table
def _load(path):
    if not os.path.exists(path): return None
    df = pd.read_csv(path)
    for c in ('patient', 'patient_id'):
        if c in df.columns:
            df = df[~df[c].astype(str).isin(['MEAN','STD'])]
    return df

rows = []
# Cross-patient LOPO
for name in ('LR', 'SVM'):
    d = _load(os.path.join(RESULTS_DIR, f'lopo_v6b_alarm_{name}.csv'))
    if d is not None:
        rows.append({'tier': '1. Cross-patient LOPO', 'config': f'V6b PDC {name}',
                     'auc': d['auc'].mean(), 'auc_pr': d['auc_pr'].mean(),
                     'sens': d['sensitivity'].mean(), 'fpr_h': d['fpr_per_hour'].mean()})

# Within-patient temporal (V8b)
for name in ('LR', 'SVM'):
    d = _load(os.path.join(RESULTS_DIR, f'lopo_v8b_alarm_{name}.csv'))
    if d is not None:
        rows.append({'tier': '2. Within-patient temporal', 'config': f'V8b PDC {name}',
                     'auc': d['auc'].mean(), 'auc_pr': d['auc_pr'].mean(),
                     'sens': d['sensitivity'].mean(), 'fpr_h': d['fpr_per_hour'].mean()})

# Patient-specific LOSO (V11)
for name, df in results_patient.items():
    rows.append({'tier': '3. Patient-specific LOSO', 'config': f'V11 PDC {name}',
                 'auc': df.auc_mean.mean(), 'auc_pr': df.auc_pr_mean.mean(),
                 'sens': df.sens_event.mean(), 'fpr_h': df.fpr_h_mean.mean()})

# Literature
rows.append({'tier': 'Literature', 'config': 'Truong 2018 (patient-spec)',
             'auc': float('nan'), 'auc_pr': float('nan'), 'sens': 0.81, 'fpr_h': 0.16})
rows.append({'tier': 'Literature', 'config': 'Khan 2018 (patient-spec)',
             'auc': float('nan'), 'auc_pr': float('nan'), 'sens': 0.875, 'fpr_h': 0.142})

cmp = pd.DataFrame(rows)
print(cmp.to_string(index=False))
cmp.to_csv(Path(RESULTS_DIR) / 'lopo_v11_three_tier_comparison.csv', index=False)
print('\nSaved lopo_v11_three_tier_comparison.csv')


                      tier                     config      auc   auc_pr     sens    fpr_h
     1. Cross-patient LOPO                 V6b PDC LR 0.570214 0.412829 0.000929 0.070610
     1. Cross-patient LOPO                V6b PDC SVM 0.582595 0.429024 0.000919 0.005600
2. Within-patient temporal                 V8b PDC LR 0.564867 0.539614 0.006738 0.294981
2. Within-patient temporal                V8b PDC SVM 0.556919 0.522710 0.007014 0.241852
  3. Patient-specific LOSO                 V11 PDC LR 0.485917 0.451485 0.419388 0.751198
  3. Patient-specific LOSO                V11 PDC SVM 0.448962 0.427169 0.406236 0.464292
                Literature Truong 2018 (patient-spec)      NaN      NaN 0.810000 0.160000
                Literature   Khan 2018 (patient-spec)      NaN      NaN 0.875000 0.142000

Saved lopo_v11_three_tier_comparison.csv
